# 🚢 Session 7 Lab: Interactive Titanic

**Prerequisite:** `01_Plotly_Tutorial.ipynb`

You drew most of these charts with Seaborn already. Now draw them in Plotly, and make the hover and the legend do some of the work.

- Use `plotly.express` (`px`) unless an exercise says otherwise.
- Every chart gets a **title** and readable labels (`labels={...}`).
- ❓ questions get a short answer in the ✍️ cell under them.
- No `for` loops.

| Part | Exercises |
|---|---|
| A: One column at a time | 1, 2, 3 |
| B: Comparing groups | 4, 5, 6 |
| C: Relationships | 7, 8 |
| D: The big picture and sharing | 9, 10, 11 |

In [1]:
!pip install plotly

In [2]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"   # plotly.js is loaded from the internet: the saved notebook stays small

df = pd.read_csv("titanic1.csv")
df.columns = df.columns.str.lower()
df["class"] = df["pclass"].map({1: "First", 2: "Second", 3: "Third"})
df["survival"] = df["survived"].map({0: "Died", 1: "Survived"})
df["port"] = df["embarked"].map({"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"})
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False)
df["age_group"] = pd.cut(df["age"], [0, 12, 18, 35, 60, 80],
                         labels=["child", "teen", "young adult", "adult", "senior"])

ORDERS = {"class": ["First", "Second", "Third"], "survival": ["Died", "Survived"],
          "age_group": ["child", "teen", "young adult", "adult", "senior"]}
COLORS = {"Died": "#D95D4A", "Survived": "#0F766E"}
df.head(3)

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,class,survival,port,family_size,title,age_group
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Third,Died,Southampton,2,Mr,young adult
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,First,Survived,Cherbourg,2,Mrs,adult
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Third,Survived,Southampton,1,Miss,young adult


The setup made the readable columns for you: `class`, `survival`, `port`, `family_size`, `title`, `age_group`, plus a fixed order (`ORDERS`) and colours (`COLORS`).

---
# A: One Column at a Time

### 1. Family size
Draw a histogram of `family_size`, one bar per size. Hover the bars: which size is the most common, and how many passengers have it?

In [3]:
# ✏️ Your code here
px.histogram(df, x = 'family_size')

❓ **Question:** Which family size is the most common, and how many passengers?

✍️ **Your answer:** _..._

### 2. Age
A histogram of `age` with a box plot on top of it.

In [4]:
# ✏️ Your code here
px.histogram(df, x = 'age',marginal="box")

> 💡 **Try it another way:** draw the same with `marginal="violin"`. What does the violin add to the box?

### 3. Fare by class
A box plot of `fare` for each `class`, each class in its own colour, classes in order. Draw it once as it is, and once on a **log** y-axis.

In [8]:
fig = px.box(df, x = 'class',y = 'fare', color = 'class', category_orders= {'class':['first', 'second', 'third']} )
fig.show()

❓ **Question:** Hover the boxes. What is the median fare in each class? Why is the log version easier to read?

✍️ **Your answer:** _..._

---
# B: Comparing Groups

### 4. The most common titles
`top_titles`: the 6 most common values of `title` with their counts. Then a bar chart of them, sorted, with the count written on each bar.

In [9]:
df.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'cabin', 'embarked', 'class', 'survival',
       'port', 'family_size', 'title', 'age_group'],
      dtype='str')

In [13]:
df['title'].value_counts(ascending=True)

title
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Major             2
Mlle              2
Col               2
Rev               6
Dr                7
Master           40
Mrs             125
Miss            182
Mr              517
Name: count, dtype: int64

In [16]:
top_titles = df['title'].value_counts(ascending=False).head(6)

# ✏️ Your code here
px.bar(top_titles)

### 5. Overall survival
A donut chart of how many passengers died and survived, in the colours of `COLORS`.

In [17]:
df.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'cabin', 'embarked', 'class', 'survival',
       'port', 'family_size', 'title', 'age_group'],
      dtype='str')

In [26]:
df['survival'].value_counts()

survival
Died        549
Survived    342
Name: count, dtype: int64

In [24]:
outcome = df['survived'].value_counts()

# ✏️ Your code here
fig = px.pie(names =['died', 'survived'], values = outcome.values,  hole=0.4)
fig.show()

### 6. Port and survival
`port_counts`: the number of passengers for each `port` and `survival`. Then a **grouped** bar chart of it, coloured by survival.

In [33]:
port_counts = df.groupby(['port', 'survival']).size().reset_index(name = 'count')

# ✏️ Your code here
px.bar(port_counts, x = 'port', y = 'count', color = 'survival', barmode = 'group')

❓ **Question:** At which port did more passengers survive than die?

✍️ **Your answer:** _..._

---
# C: Relationships

### 7. Age, fare and class
A scatter of `age` (x) against `fare` (y):
- coloured by `survival`, one panel per `class` (in order), fare on a log axis
- the hover shows the passenger's **name** in bold, plus `sex` and `family_size`
- the panel titles say only *First*, *Second*, *Third*

In [34]:
# ✏️ Your code here
px.scatter(df, x = 'age', y = 'fare', color = 'survival')

❓ **Question:** In which panel are the survivors the majority? Hover one of the oldest survivors: who is it?

✍️ **Your answer:** _..._

### 8. Survival rate heatmap
`rate`: a table of the survival rate, `sex` as rows and `class` as columns. Show it as a heatmap with the rates written in the cells as percentages, and the colour scale from 0 to 1.

In [45]:
rate = (df.groupby(['sex', 'class'])['survived'].mean() ).reset_index()
print(rate)
# ✏️ Your code here
px.density_heatmap(rate, x = 'class', y = 'sex', z = 'survived', text_auto='0.1%', title = 'Survival Rate by Sex and Class')

      sex   class  survived
0  female   First  0.968085
1  female  Second  0.921053
2  female   Third  0.500000
3    male   First  0.368852
4    male  Second  0.157407
5    male   Third  0.135447


> 💡 **Try it another way:** the same heatmap with `age_group` as rows. And a correlation heatmap of the numeric columns, colour scale from −1 to 1.

❓ **Question:** Which group had the highest survival rate, and which the lowest?

✍️ **Your answer:** _..._

---
# D: The Big Picture and Sharing

### 9. Passenger treemap
`groups`: one row per `class` and `sex` with the number of passengers (`passengers`) and the survival rate (`rate`). Then a treemap class → sex: the box size is the number of passengers, the colour is the survival rate.

In [ ]:
groups = ...

# ✏️ Your code here
...

> 💡 **Try it another way:** the same `groups` as a `px.sunburst`.

### 10. Titles as bubbles
For the titles `Mr`, `Mrs`, `Miss`, `Master`: `titles` holds the mean age, the survival rate and the number of passengers of each. Then a bubble chart: x = mean age, y = survival rate, size = passengers, colour = title.

In [ ]:
titles = ...

# ✏️ Your code here
...

❓ **Question:** *Master* were boys. How does their survival rate compare with *Mr*?

✍️ **Your answer:** _..._

### 11. Share it
Save the treemap from exercise 9 as an interactive web page `figures/titanic_treemap.html`. Then list the files in `figures/` with their size.

In [ ]:
# ✏️ Your code here
...

❓ **Question:** When would you send a Plotly HTML file instead of a Seaborn PNG?

✍️ **Your answer:** _..._

---
# 🔭 Try After the Session
Add the 💡 ideas you skipped. Then take chart 7 and add `animation_frame="class"` instead of `facet_col`: is a slider or three side-by-side panels easier to compare?